#### DATA INGESTION - DATAFRAME

##### 01.1 DOCUMENTAÇÃO DO DATA INGESTION

###### Objetivo

Disponibilizar um DataFrame PySpark para as etapas de preparação, análise e avaliação de qualidade dos dados, garantindo que o framework seja independente da origem e do formato da fonte.

O Data Ingestion é responsável exclusivamente pela configuração, leitura, validação estrutural e registro de metadados da fonte, sem aplicar regras de negócio ou transformações analíticas.

---

###### Fonte de dados utilizada

Para demonstração e validação do framework, utiliza-se o conjunto de dados **Ocorrências Emergenciais da Rede de Distribuição — 2026**, disponibilizado pela **ANEEL — Agência Nacional de Energia Elétrica**.

A fonte é utilizada apenas como exemplo, não constituindo uma dependência obrigatória do framework.

---

###### Arquitetura genérica

O componente permite utilizar diferentes fontes de dados compatíveis com Apache Spark:

- **CSV:** arquivos delimitados, com opções configuráveis de leitura.
- **Parquet:** arquivos colunares com schema incorporado.
- **Delta:** dados armazenados no formato Delta Lake.
- **Tabelas Spark:** tabelas registradas no catálogo.

A substituição da fonte deve ocorrer por meio dos parâmetros de configuração, sem necessidade de modificar os Profiles analíticos.

---

###### Principais etapas

1. **Configuração:** definição do tipo de fonte, localização e opções de leitura.
2. **Validação dos parâmetros:** verificação das configurações antes do carregamento.
3. **Leitura:** carregamento da fonte utilizando a API nativa do PySpark.
4. **Validação estrutural:** verificação do tipo do objeto, existência de colunas e identificação de nomes inválidos ou duplicados.
5. **Metadados:** registro das características técnicas da fonte e do schema.
6. **Visualização:** apresentação opcional de uma amostra limitada de registros.
7. **Testes:** execução opcional de testes sintéticos do contrato de ingestão.

---

###### Contrato de entrada

O componente recebe os seguintes parâmetros:

| Parâmetro | Descrição |
|---|---|
| `INGESTION_SOURCE_TYPE` | Formato ou tipo da fonte de dados. |
| `INGESTION_SOURCE` | Caminho do arquivo ou identificador da tabela. |
| `INGESTION_OPTIONS` | Opções específicas de leitura. |
| `INGESTION_SCHEMA` | Schema explícito opcional. |
| `INGESTION_DISPLAY_RESULTS` | Habilita a visualização dos dados. |
| `INGESTION_PREVIEW_ROWS` | Limita a quantidade de registros exibidos. |
| `INGESTION_RUN_TESTS` | Habilita os testes do contrato de ingestão. |

---

###### Contrato de saída

**DataFrame principal: `df`**

A saída deve atender às seguintes condições:

- Ser uma instância de `pyspark.sql.DataFrame`.
- Possuir pelo menos uma coluna.
- Apresentar nomes de colunas válidos e não duplicados.
- Preservar a estrutura e os tipos resultantes da leitura.
- Permanecer disponível para o Notebook 02 — Data Preparation.

**DataFrame auxiliar: `ingestion_metadata_df`**

Registra informações técnicas da ingestão, como:

- Identificação da fonte;
- Formato dos dados;
- Nome das colunas;
- Tipos Spark;
- Nulabilidade;
- Informações de execução.

---

###### Tratamento de valores especiais

O componente não modifica valores nulos, vazios, duplicados ou atípicos encontrados na fonte.

Essas características são preservadas para análise pelos Profiles especializados.

O tratamento de situações excepcionais contempla:

- Formatos de leitura não suportados;
- Parâmetros inválidos;
- Fontes inexistentes ou inacessíveis;
- Falhas durante a leitura;
- DataFrames sem colunas;
- Nomes de colunas inválidos ou duplicados.

A validação da existência de registros permanece sob responsabilidade do Notebook 02 — Data Preparation.

---

###### Características do componente

- **Genérico:** não depende de nomes de colunas ou regras de domínio.
- **Parametrizável:** permite configurar diferentes fontes.
- **Reutilizável:** utiliza um contrato de saída padronizado.
- **Escalável:** utiliza a API distribuída do PySpark.
- **Não intrusivo:** não altera o conteúdo original dos dados.
- **Rastreável:** registra metadados técnicos.
- **Testável:** possui verificações estruturais independentes da fonte.
- **Integrável:** disponibiliza o DataFrame para os demais notebooks.

---

###### DataFrames de resultado

| DataFrame | Finalidade | Granularidade |
|---|---|---|
| `df` | Dados carregados da fonte. | Um registro por linha da fonte, conforme interpretação do leitor Spark. |
| `ingestion_metadata_df` | Metadados técnicos da ingestão. | Uma linha por coluna do DataFrame. |

---

###### Resultado esperado

Disponibilizar um DataFrame PySpark estruturalmente válido, acompanhado de metadados técnicos, para utilização pelas etapas seguintes do Data Quality Framework.

A ingestão deve permanecer independente das regras analíticas e permitir a substituição da fonte de dados por meio de configurações.

> **Princípio arquitetural:** a origem dos dados deve ser substituível sem exigir alterações na lógica dos Profiles de qualidade.


In [0]:
%run "../bibliotecas/00_FRAMEWORK_CONFIG"

In [0]:
# ============================================================
# 01.3 CONFIGURAÇÃO DA FONTE DE DADOS
# ============================================================
#
# O QUE FAZ:
# Define origem, formato e opções de leitura.
#
# COMO FAZ:
# Centraliza parâmetros editáveis antes da leitura Spark.
#
# POR QUE É IMPORTANTE:
# Permite substituir a fonte sem modificar os Profiles.
#
# PERGUNTA RESPONDIDA:
# "Qual fonte será disponibilizada ao framework?"

INGESTION_SOURCE_TYPE = "csv"  # csv | parquet | delta | table
INGESTION_SOURCE = (
    "/Volumes/setor_eletrico/distribuicao/"
    "ocorrencias_emergenciais_setor_eletrico/"
    "ocorrencias-emergenciais-rede-distribuicao-2026.csv"
)

# As opções abaixo são específicas da fonte CSV de demonstração.
# Para Parquet, Delta ou table, utilize {} salvo necessidade explícita.
INGESTION_OPTIONS = {
    "header": "true",
    "delimiter": ";",
    "quote": '"',
    "inferSchema": "true",
}

# Para leitura com schema fixo, informe um StructType.
# Exemplo: T.StructType([T.StructField("codigo", T.StringType(), True)])
INGESTION_SCHEMA = None

INGESTION_DISPLAY_RESULTS = False
INGESTION_PREVIEW_ROWS = 10
INGESTION_RUN_TESTS = False
INGESTION_SUPPORTED_FORMATS = ("csv", "parquet", "delta", "table")

In [0]:
# ============================================================
# 01.4 VALIDAÇÃO DOS PARÂMETROS
# ============================================================
#
# O QUE FAZ:
# Rejeita configurações inválidas antes da leitura.
#
# COMO FAZ:
# Valida tipo de fonte, identificador, opções e schema.
#
# POR QUE É IMPORTANTE:
# Evita erros de configuração difíceis de diagnosticar.
#
# PERGUNTA RESPONDIDA:
# "Os parâmetros de ingestão são válidos?"

def dq_ingestion_validate_config(source_type, source, options, schema):
    if source_type not in INGESTION_SUPPORTED_FORMATS:
        raise ValueError(f"Formato não suportado: {source_type!r}.")

    if not isinstance(source, str) or not source.strip():
        raise ValueError("INGESTION_SOURCE deve ser uma string não vazia.")

    if not isinstance(options, dict):
        raise TypeError("INGESTION_OPTIONS deve ser um dicionário.")

    if any(not isinstance(k, str) or not isinstance(v, str)
           for k, v in options.items()):
        raise TypeError("As opções Spark devem possuir chaves e valores string.")

    if schema is not None and not isinstance(schema, T.StructType):
        raise TypeError("INGESTION_SCHEMA deve ser StructType ou None.")

    if source_type == "table" and (options or schema is not None):
        raise ValueError("Para 'table', utilize INGESTION_OPTIONS={} e INGESTION_SCHEMA=None.")

    if schema is not None and source_type == "csv" and "inferSchema" in options:
        raise ValueError("Com schema explícito, remova a opção 'inferSchema' do CSV.")

    return True


dq_ingestion_validate_config(
    INGESTION_SOURCE_TYPE,
    INGESTION_SOURCE,
    INGESTION_OPTIONS,
    INGESTION_SCHEMA,
)

if not isinstance(INGESTION_DISPLAY_RESULTS, bool):
    raise TypeError("INGESTION_DISPLAY_RESULTS deve ser booleano.")
if not isinstance(INGESTION_RUN_TESTS, bool):
    raise TypeError("INGESTION_RUN_TESTS deve ser booleano.")
if isinstance(INGESTION_PREVIEW_ROWS, bool) or not isinstance(INGESTION_PREVIEW_ROWS, int) or INGESTION_PREVIEW_ROWS <= 0:
    raise ValueError("INGESTION_PREVIEW_ROWS deve ser inteiro positivo.")

print("Configurações de ingestão validadas.")

In [0]:
# ============================================================
# 01.5 LEITURA GENÉRICA COM PYSPARK
# ============================================================
#
# O QUE FAZ:
# Lê a fonte configurada e disponibiliza o DataFrame df.
#
# COMO FAZ:
# Utiliza spark.table() ou spark.read.format().load().
#
# POR QUE É IMPORTANTE:
# Mantém o contrato de saída independente da origem.
#
# PERGUNTA RESPONDIDA:
# "Como carregar fontes diferentes com a mesma interface?"

def dq_ingestion_read(spark_session, source_type, source, options=None, schema=None):
    dq_ingestion_validate_config(source_type, source, options or {}, schema)

    if source_type == "table":
        return spark_session.table(source)

    reader = spark_session.read.format(source_type)
    if options:
        reader = reader.options(**options)
    if schema is not None:
        reader = reader.schema(schema)
    return reader.load(source)


try:
    df = dq_ingestion_read(
        spark_session=spark,
        source_type=INGESTION_SOURCE_TYPE,
        source=INGESTION_SOURCE,
        options=INGESTION_OPTIONS,
        schema=INGESTION_SCHEMA,
    )
except Exception as exc:
    raise RuntimeError(
        f"Falha na ingestão: tipo={INGESTION_SOURCE_TYPE!r}, "
        f"origem={INGESTION_SOURCE!r}, "
        f"erro={type(exc).__name__}: {exc}"
    ) from exc

print("DataFrame Spark disponibilizado em 'df'.")

In [0]:
# ============================================================
# 01.6 VALIDAÇÃO DO CONTRATO DE SAÍDA
# ============================================================
#
# O QUE FAZ:
# Verifica se a ingestão produziu um DataFrame válido.
#
# COMO FAZ:
# Inspeciona schema sem contar linhas do dataset.
#
# POR QUE É IMPORTANTE:
# Evita propagação de estrutura inválida ao Notebook 02.
#
# PERGUNTA RESPONDIDA:
# "O DataFrame atende ao contrato estrutural mínimo?"

from pyspark.sql import DataFrame

def dq_ingestion_validate_dataframe(dataframe):
    if not isinstance(dataframe, DataFrame):
        raise TypeError("A saída deve ser um DataFrame Spark.")

    column_names = [field.name for field in dataframe.schema.fields]
    if not column_names:
        raise ValueError("O DataFrame não possui colunas.")
    if any(not name.strip() for name in column_names):
        raise ValueError("O DataFrame possui nome de coluna vazio.")
    if len(column_names) != len(set(column_names)):
        raise ValueError("O DataFrame possui nomes de colunas duplicados.")
    return True


dq_ingestion_validate_dataframe(df)
print(f"Contrato estrutural validado: {len(df.columns)} colunas.")

In [0]:
# ============================================================
# 01.7 METADADOS DA INGESTÃO EM DATAFRAME SPARK
# ============================================================
#
# O QUE FAZ:
# Registra origem, formato e schema observado por coluna.
#
# COMO FAZ:
# Constrói um DataFrame Spark a partir de metadados do schema.
#
# POR QUE É IMPORTANTE:
# Fornece rastreabilidade sem coletar registros da fonte.
#
# PERGUNTA RESPONDIDA:
# "Quais colunas e tipos foram disponibilizados?"

INGESTION_METADATA_SCHEMA = T.StructType([
    T.StructField("source_type", T.StringType(), False),
    T.StructField("source", T.StringType(), False),
    T.StructField("column_position", T.IntegerType(), False),
    T.StructField("column_name", T.StringType(), False),
    T.StructField("spark_type", T.StringType(), False),
    T.StructField("nullable", T.BooleanType(), False),
    T.StructField("ingested_at_utc", T.TimestampType(), False),
])

# O timestamp representa a criação do inventário técnico, não a
# materialização integral do dataset.
ingestion_timestamp = datetime.now(timezone.utc).replace(tzinfo=None)

ingestion_metadata_rows = [
    (
        INGESTION_SOURCE_TYPE,
        INGESTION_SOURCE,
        position,
        field.name,
        field.dataType.simpleString(),
        bool(field.nullable),
        ingestion_timestamp,
    )
    for position, field in enumerate(df.schema.fields)
]

ingestion_metadata_df = spark.createDataFrame(
    ingestion_metadata_rows,
    schema=INGESTION_METADATA_SCHEMA,
)

print("Metadados técnicos disponibilizados em 'ingestion_metadata_df'.")

In [0]:

# ============================================================
# 01.8 VISUALIZAÇÃO OPCIONAL
# ============================================================
#
# O QUE FAZ:
# Apresenta uma amostra dos dados carregados.
#
# COMO FAZ:
# Utiliza operações nativas do PySpark para limitar
# os registros exibidos.
#
# POR QUE É IMPORTANTE:
# Permite inspecionar a fonte sem exigir visualização
# durante as execuções automatizadas.
#
# PERGUNTA RESPONDIDA:
# "Como visualizar os dados de forma controlada?"

if INGESTION_DISPLAY_RESULTS:

    print("VISUALIZAÇÃO DA FONTE DE DADOS")
    print("-" * 50)
    print(f"Fonte: {INGESTION_SOURCE}")
    print(f"Limite de registros: {INGESTION_PREVIEW_ROWS}")

    display(
        df.limit(INGESTION_PREVIEW_ROWS)
    )

    print("Visualização concluída com sucesso.")

else:

    print(
        "Visualização desabilitada. "
        "Defina INGESTION_DISPLAY_RESULTS = True "
        "na célula 01.3 para habilitar."
    )

In [0]:

# ============================================================
# 01.9 TESTES DO CONTRATO DE INGESTÃO
# ============================================================
#
# O QUE FAZ:
# Valida o comportamento estrutural da ingestão.
#
# COMO FAZ:
# Executa testes sintéticos com DataFrames PySpark,
# sem utilizar RDDs ou pandas.
#
# POR QUE É IMPORTANTE:
# Verifica se o contrato aceita estruturas válidas
# e rejeita estruturas inválidas.
#
# PERGUNTA RESPONDIDA:
# "O contrato de ingestão está funcionando corretamente?"

if INGESTION_RUN_TESTS:

    print("INICIANDO TESTES DO DATA INGESTION")
    print("=" * 50)

    # --------------------------------------------------------
    # TESTE 01 - DATAFRAME VÁLIDO
    # --------------------------------------------------------

    teste_valido_df = spark.range(5).select(
        F.col("id").alias("codigo")
    )

    assert dq_ingestion_validate_dataframe(
        teste_valido_df
    )

    print("[OK] Teste 01 - DataFrame válido.")

    # --------------------------------------------------------
    # TESTE 02 - DATAFRAME SEM COLUNAS
    # --------------------------------------------------------

    teste_sem_colunas_df = spark.range(1).select()

    try:
        dq_ingestion_validate_dataframe(
            teste_sem_colunas_df
        )

    except ValueError:
        print("[OK] Teste 02 - Sem colunas rejeitado.")

    else:
        raise AssertionError(
            "O validador aceitou um DataFrame sem colunas."
        )

    # --------------------------------------------------------
    # TESTE 03 - OBJETO INVÁLIDO
    # --------------------------------------------------------

    try:
        dq_ingestion_validate_dataframe(None)

    except TypeError:
        print("[OK] Teste 03 - Objeto inválido rejeitado.")

    else:
        raise AssertionError(
            "O validador aceitou um objeto inválido."
        )

    # --------------------------------------------------------
    # TESTE 04 - NOMES DUPLICADOS
    # --------------------------------------------------------

    teste_duplicado_df = spark.range(1).select(
        F.col("id").alias("codigo"),
        F.col("id").alias("codigo")
    )

    try:
        dq_ingestion_validate_dataframe(
            teste_duplicado_df
        )

    except ValueError:
        print("[OK] Teste 04 - Nomes duplicados rejeitados.")

    else:
        raise AssertionError(
            "O validador aceitou colunas duplicadas."
        )

    # --------------------------------------------------------
    # RESULTADO FINAL
    # --------------------------------------------------------

    print("=" * 50)
    print("RESULTADO: 4 TESTES APROVADOS.")
    print("Contrato de ingestão validado com sucesso.")

else:

    print(
        "Testes desabilitados. "
        "Defina INGESTION_RUN_TESTS = True "
        "na célula 01.3 para executar."
    )